In [1]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

In [2]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-base'
    tokenizer = None
    max_length = 128
    batch_size = 1
    fc_dropout = 0.0
    model_config = {
        'attention_dropout': 0.0,
        'attention_probs_dropout_prob': 0.0,
        'hidden_dropout': 0.0,
        'hidden_dropout_prob': 0.0,
    }
    target_size = 1,
    target_cols = 'score'
    max_len = 1024,
    seed = 42
    n_fold = 6
    trn_fold = [0, 1, 2, 3, 4, 5]
    head = 'mean_pooling'
    epochs=10


In [3]:
def get_score(y_acts, y_preds):
    score = cohen_kappa_score(y_acts, y_preds, weights='quadratic')
    return score

In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(seed=42)

In [5]:
path = kagglehub.competition_download(cfg.competition)

In [6]:
train_df = pd.read_csv(f'{path}/train.csv')
test_df = pd.read_csv(f'{path}/test.csv')

In [7]:
display(train_df.head())
display(test_df.head())

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


,essay_id,full_text
0,000d118,Many people have car where they live. The thin...
1,000fe60,I am a scientist at NASA that is discussing th...
2,001ab80,People always wish they had the same technolog...


In [8]:
train_df.columns

Index(['essay_id', 'full_text', 'score'], dtype='str')

In [9]:
train_df.score.describe()

count    17307.000000
mean         2.948402
std          1.044899
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max          6.000000
Name: score, dtype: float64

In [10]:
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1
train_df['score'].describe()

count    17307.000000
mean         1.948402
std          1.044899
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          5.000000
Name: score, dtype: float64

In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
                                train_df['full_text'], 
                                train_df['score'],
                                test_size=0.2,
                                stratify=train_df['score'].astype(int),
                                random_state=42                       
                                )

In [12]:
train_df = pd.DataFrame({
    'full_text': X_train,
    'score': y_train
})
valid_df = pd.DataFrame({
    'full_text': X_valid,
    'score': y_valid
})

In [13]:
print(len(X_train))
print(len(X_valid))
print(type(X_train.iloc[0]))
print(type(y_train.iloc[0]))
#Check validation split and data types

13845
3462
<class 'str'>
<class 'numpy.int64'>


In [14]:
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)
tokenized = tokenizer(X_train.iloc[0], max_length=512, padding = 'max_length')

tokenized2 = tokenizer(X_train.iloc[1], max_length=512, padding = 'max_length')


print(len(tokenized['input_ids']))
print(len(tokenized2['input_ids']))

print(tokenized['input_ids'][0])
print(tokenized['input_ids'][-1])

print(tokenized2['input_ids'][0])
print(tokenized2['input_ids'][-1])

#Truncates longer essays to 512 tokens and pads shorter ones


800
512
1
2
1
0


In [15]:
from torch.utils.data import Dataset, DataLoader
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512 ):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.long)
        return item


In [16]:
#Test on sample
dataset = EssayDataset(train_df, tokenizer)
sample = dataset[0]
sample.keys()
sample['input_ids'].shape
sample['attention_mask'].shape
sample['labels']
sample['labels'].dtype

torch.int64

In [17]:
# train_dataset = EssayDataset(train_df, tokenizer)
# valid_dataset = EssayDataset(valid_df, tokenizer)

# train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle = True)
# valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle = False)

In [18]:
# batch = next(iter(train_loader))
# display(batch['input_ids'].shape)
# display(batch['attention_mask'].shape)
# display(batch['labels'].shape)

In [19]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [20]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Linear(self.model.config.hidden_size, 6)

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [21]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
deberta_model = EssayModel().to(device)
print(next(deberta_model.parameters()).device)

True


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


mps:0


In [22]:
# #Test one optimization
# optimizer = torch.optim.SGD(params=deberta_model.parameters(), lr=5e-5)

# optimizer.zero_grad()
# batch = next(iter(train_loader))
# logits_1 = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
# loss_fn = nn.CrossEntropyLoss()
# loss_1 = loss_fn(logits_1, batch['labels'].to(device))
# print('loss before:', loss_1)
# loss_1.backward()

# optimizer.step()
# optimizer.zero_grad()
# logits_2 = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
# print(logits_2)
# loss_2 = loss_fn(logits_2, batch['labels'].to(device))
# print('loss after:', loss_2)
# print(next(deberta_model.parameters()).device)


In [ ]:
DEBUG = True
if DEBUG:
    train_df_used = train_df.sample(frac=0.01, random_state=42)
    valid_df_used = valid_df.sample(frac=0.01, random_state=42)
else:
    train_df_used = train_df
    valid_df_used = valid_df
    
train_dataset = EssayDataset(train_df_used, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
valid_dataset = EssayDataset(valid_df_used, tokenizer)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False)

test_dataset = EssayDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle = False)

In [24]:
optimizer = torch.optim.AdamW(
    deberta_model.parameters(), 
    lr=1e-6,
    eps=1e-6,
    )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

loss_fn = nn.CrossEntropyLoss()

#Store best QWK
best_kappa = -999

for epoch in range(cfg.epochs):
    #Train
    deberta_model.train()
    train_loss = 0; train_correct = 0; train_total = 0
    for batch in train_loader:
        optimizer.zero_grad()
        labels = batch['labels'].to(device)
        logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = loss_fn(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            deberta_model.parameters(),
            max_norm=1.0
        )
        optimizer.step()
    avg_train_loss = train_loss/len(train_loader)
    train_accuracy = train_correct / train_total

    #Valid
    deberta_model.eval()
    valid_loss = 0; valid_correct = 0; valid_total = 0
    all_labels = []; all_preds = []
    with torch.no_grad():
        for batch in valid_loader:
            labels = batch['labels'].to(device)
            logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = loss_fn(logits, labels)
            valid_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            valid_correct += (preds==labels).sum().item()
            valid_total += labels.size(0)
    avg_valid_loss = valid_loss/len(valid_loader)
    valid_accuracy = valid_correct/valid_total
    kappa = get_score(all_labels, all_preds)
    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(
            deberta_model.state_dict(),
            'best_deberta_model.pth'
        )
        print(f'Saved new-best model: QWK {best_kappa:.4f}')

    scheduler.step()

    #Summary
    print(f'Epoch {epoch+1}/{cfg.epochs}')
    print(f'Train Loss: {avg_train_loss:.4f}')
    print(f'Train Accuracy: {train_accuracy:.4f}')
    print(f'Valid Loss:{avg_valid_loss:.4f}')
    print(f'Valid Accuracy: {valid_accuracy:.4f}')
    print(f'Quadratic Weighted Kappa: {kappa:.4f}')
    print(f'Best QWK So Far: {best_kappa:.4f}')
    print(f'LR: {scheduler.get_last_lr()[0]:.2e}\n')

Saved new-best model: QWK 0.0000
Epoch 1/10
Train Loss: 1.6163
Train Accuracy: 0.3116
Valid Loss:1.5214
Valid Accuracy: 0.2000
Quadratic Weighted Kappa: 0.0000
Best QWK So Far: 0.0000
LR: 9.76e-07

Epoch 2/10
Train Loss: 1.4549
Train Accuracy: 0.3913
Valid Loss:1.3775
Valid Accuracy: 0.2571
Quadratic Weighted Kappa: 0.0000
Best QWK So Far: 0.0000
LR: 9.05e-07

Epoch 3/10
Train Loss: 1.4841
Train Accuracy: 0.3116
Valid Loss:1.5580
Valid Accuracy: 0.2571
Quadratic Weighted Kappa: 0.0000
Best QWK So Far: 0.0000
LR: 7.94e-07

Epoch 4/10
Train Loss: 1.4438
Train Accuracy: 0.3768
Valid Loss:1.4231
Valid Accuracy: 0.2571
Quadratic Weighted Kappa: 0.0000
Best QWK So Far: 0.0000
LR: 6.55e-07



KeyboardInterrupt: 

In [ ]:
#Test
#Load best model
deberta_model.load_state_dict(
    torch.load('best_deberta_model.pth')
)
deberta_model.eval()
test_preds = []

with torch.no_grad():
    for batch in test_loader:
        logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())

#Add 1 back to scores
test_preds = [p+1 for p in test_preds]

#Submission
submission = pd.DataFrame({
    'essay_id': test_df['essay_id'],
    'score': test_preds
})
submission.to_csv('submission.csv', index=False)